# 객체 탐지 YOLO — Global Wheat Detection (ultralytics) — Colab

[Global Wheat Detection](https://www.kaggle.com/c/global-wheat-detection) (밀 이삭 탐지, 단일 클래스 바운딩박스) 를 **YOLO(ultralytics)** 로 푸는 객체 탐지 기본 예제입니다.

- 핵심 흐름: **bbox 어노테이션 → YOLO 포맷 변환 → `data.yaml` → 학습 → 추론 → 제출**.
- 사전학습 `yolo11n` 을 미세조정합니다.
- 실행 시 **Kaggle API로 데이터를 직접 다운로드**하며, 토큰이 **노트북에 내장**되어 바로 실행됩니다.
- 마지막에 제출 파일 `submission.csv` (`image_id,PredictionString`) 가 `/content` 에 생성됩니다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU (객체 탐지 학습은 CPU에서 매우 느림).  
> ⚠️ **사전 필수**: `jinusun` 계정으로 [대회 규칙](https://www.kaggle.com/c/global-wheat-detection/rules) Join 필요.  
> ℹ️ 이 대회는 코드 대회라 공개 test 는 샘플 약 10장입니다(실제 채점셋은 숨김). 학습은 전체 데이터로 가능합니다.  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "ultralytics", "pandas", "numpy", "pillow", "pyyaml"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Global Wheat Detection 데이터 다운로드

> 약 600MB라 수 분 걸릴 수 있습니다.

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("global-wheat-detection", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("내용:", [f for f in sorted(os.listdir(WORK_DIR)) if not f.startswith(".")][:10])
print("train images:", len(os.listdir(os.path.join(WORK_DIR, "train"))),
      "| test images:", len(os.listdir(os.path.join(WORK_DIR, "test"))))

## 2. 어노테이션 → YOLO 포맷 변환

`train.csv` 의 bbox `[x, y, w, h]`(좌상단+크기, 픽셀)를 YOLO 포맷 `class x_center y_center w h`(0~1 정규화)로 변환해 이미지별 `.txt` 라벨을 만듭니다. (이미지 1024×1024, 단일 클래스)

In [ ]:
import pandas as pd, numpy as np, ast, shutil

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
df["bbox"] = df["bbox"].apply(ast.literal_eval)
print("annotated images:", df["image_id"].nunique(), "| boxes:", len(df))

SIZE = 1024  # 원본 이미지 크기
YROOT = os.path.join(WORK_DIR, "wheat_yolo")
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(os.path.join(YROOT, sub), exist_ok=True)

image_ids = df["image_id"].unique()
rng = np.random.RandomState(42); rng.shuffle(image_ids)
cut = int(len(image_ids) * 0.9)
split = {i: ("train" if k < cut else "val") for k, i in enumerate(image_ids)}

for image_id, g in df.groupby("image_id"):
    s = split[image_id]
    shutil.copy(os.path.join(WORK_DIR, "train", image_id + ".jpg"),
                os.path.join(YROOT, f"images/{s}", image_id + ".jpg"))
    lines = []
    for x, y, w, h in g["bbox"]:
        xc = (x + w / 2) / SIZE; yc = (y + h / 2) / SIZE
        lines.append(f"0 {xc:.6f} {yc:.6f} {w/SIZE:.6f} {h/SIZE:.6f}")
    with open(os.path.join(YROOT, f"labels/{s}", image_id + ".txt"), "w") as f:
        f.write("\n".join(lines))
print("변환 완료 ->", YROOT)

## 3. data.yaml 작성

In [ ]:
import yaml
data_yaml = os.path.join(YROOT, "data.yaml")
with open(data_yaml, "w") as f:
    yaml.safe_dump({
        "path": YROOT,
        "train": "images/train",
        "val": "images/val",
        "nc": 1,
        "names": ["wheat"],
    }, f)
print(open(data_yaml).read())

## 4. YOLO 학습

사전학습 `yolo11n` 을 미세조정합니다. (Colab GPU 기준 epochs 를 늘리세요)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # 사전학습 nano 모델
model.train(data=data_yaml, epochs=15, imgsz=640, batch=16, project=os.path.join(WORK_DIR, "runs"), name="wheat")

## 5. 테스트 추론 & 제출 파일 생성

각 테스트 이미지에 대해 박스를 예측하고, 대회 포맷 `PredictionString = "conf x y w h ..."`(좌상단+크기, 픽셀) 로 만듭니다.

In [ ]:
import glob
TEST_DIR = os.path.join(WORK_DIR, "test")
test_imgs = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")))

rows = []
for path in test_imgs:
    image_id = os.path.splitext(os.path.basename(path))[0]
    res = model.predict(path, conf=0.25, imgsz=640, verbose=False)[0]
    preds = []
    for box, conf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = box
        preds.append(f"{conf:.4f} {int(x1)} {int(y1)} {int(x2 - x1)} {int(y2 - y1)}")
    rows.append({"image_id": image_id, "PredictionString": " ".join(preds)})

submission_path = os.path.join(WORK_DIR, "submission.csv")
sub = pd.DataFrame(rows, columns=["image_id", "PredictionString"])
sub.to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(sub))
sub.head()

## 6. 제출 파일 내려받기

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[Global Wheat Detection 제출 페이지](https://www.kaggle.com/c/global-wheat-detection/submit)**

- 대회 홈: https://www.kaggle.com/c/global-wheat-detection
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.